# 02 — Silver: normalize and validate

Reads the latest successful Bronze batch, applies explicit schemas and canonical state/fuel
mappings, preserves suppressed and missing values as null plus a status, and materializes
validated Delta tables.

Natural keys:

* Operational: `(period, state_code, fuel_code)`
* Retail sales: `(period, state_code)`
* Population: `(state_code)`

In [ ]:
environment = "dev"
expected_jurisdictions = 51

In [ ]:
from pyspark.sql import functions as F, types as T, Window

if environment not in {"dev", "test", "prod"}:
    raise ValueError("environment must be dev, test, or prod")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

raw = spark.table("bronze.energy_source_raw")
latest_batch = (
    raw.groupBy("batch_id")
    .agg(F.max("ingested_at_utc").alias("ingested_at_utc"))
    .orderBy(F.desc("ingested_at_utc"))
    .first()
)
if latest_batch is None:
    raise RuntimeError("Bronze is empty; run 01_bronze_ingest first.")
batch_id = latest_batch["batch_id"]
raw = raw.where(F.col("batch_id") == batch_id)

In [ ]:
STATE_ROWS = [
    ("01","AL","Alabama"),("02","AK","Alaska"),("04","AZ","Arizona"),("05","AR","Arkansas"),
    ("06","CA","California"),("08","CO","Colorado"),("09","CT","Connecticut"),("10","DE","Delaware"),
    ("11","DC","District of Columbia"),("12","FL","Florida"),("13","GA","Georgia"),("15","HI","Hawaii"),
    ("16","ID","Idaho"),("17","IL","Illinois"),("18","IN","Indiana"),("19","IA","Iowa"),
    ("20","KS","Kansas"),("21","KY","Kentucky"),("22","LA","Louisiana"),("23","ME","Maine"),
    ("24","MD","Maryland"),("25","MA","Massachusetts"),("26","MI","Michigan"),("27","MN","Minnesota"),
    ("28","MS","Mississippi"),("29","MO","Missouri"),("30","MT","Montana"),("31","NE","Nebraska"),
    ("32","NV","Nevada"),("33","NH","New Hampshire"),("34","NJ","New Jersey"),("35","NM","New Mexico"),
    ("36","NY","New York"),("37","NC","North Carolina"),("38","ND","North Dakota"),("39","OH","Ohio"),
    ("40","OK","Oklahoma"),("41","OR","Oregon"),("42","PA","Pennsylvania"),("44","RI","Rhode Island"),
    ("45","SC","South Carolina"),("46","SD","South Dakota"),("47","TN","Tennessee"),("48","TX","Texas"),
    ("49","UT","Utah"),("50","VT","Vermont"),("51","VA","Virginia"),("53","WA","Washington"),
    ("54","WV","West Virginia"),("55","WI","Wisconsin"),("56","WY","Wyoming"),
]
state_schema = "census_state_fips string, state_code string, state_name string"
states = spark.createDataFrame(STATE_ROWS, state_schema)

FUEL_CATEGORY = {
    "SUN": "carbon_free", "WND": "carbon_free", "HYC": "carbon_free",
    "HPS": "carbon_free", "NUC": "carbon_free", "GEO": "carbon_free",
    "WWW": "carbon_free", "WAS": "carbon_free",
    "COL": "fossil", "NG": "fossil", "PEL": "fossil", "PC": "fossil",
    "OOG": "fossil", "SGC": "fossil",
}
fuel_category_expr = F.create_map(
    *[item for pair in FUEL_CATEGORY.items() for item in (F.lit(pair[0]), F.lit(pair[1]))]
)

suppressed = ["--", "NA", "N/A", "NM", "W", "S", "*"]

def numeric_status(column_name):
    raw_value = F.trim(F.col(column_name))
    return (
        F.when(F.col(column_name).isNull() | (raw_value == ""), F.lit("missing"))
        .when(F.upper(raw_value).isin(suppressed), F.lit("suppressed"))
        .otherwise(F.lit("reported"))
    )

def numeric_value(column_name):
    return (
        F.when(numeric_status(column_name) == "reported", F.regexp_replace(F.col(column_name), ",", "").cast("double"))
        .otherwise(F.lit(None).cast("double"))
    )

In [ ]:
operational = (
    raw.where(F.col("dataset_name") == "eia_electric_power_operational")
    .withColumn("state_code", F.upper(F.trim("state_code_raw")))
    .withColumn("fuel_code", F.upper(F.trim("fuel_code_raw")))
    .withColumn("generation_mwh", numeric_value("generation_raw"))
    .withColumn("generation_status", numeric_status("generation_raw"))
    .withColumn("total_consumption", numeric_value("total_consumption_raw"))
    .withColumn("consumption_for_eg", numeric_value("consumption_for_eg_raw"))
    .withColumn("total_consumption_btu", numeric_value("total_consumption_btu_raw"))
    .withColumn("consumption_for_eg_btu", numeric_value("consumption_for_eg_btu_raw"))
    .join(F.broadcast(states), "state_code", "inner")
    .withColumn("fuel_category", F.coalesce(fuel_category_expr[F.col("fuel_code")], F.lit("other")))
    .select(
        "period", "state_code", "state_name", "fuel_code",
        F.coalesce(F.col("fuel_name_raw"), F.col("fuel_code")).alias("fuel_name"),
        "fuel_category", "generation_mwh", "generation_status", "total_consumption",
        "consumption_for_eg", "total_consumption_btu", "consumption_for_eg_btu",
        "generation_units_raw", "consumption_for_eg_btu_units_raw",
        "source_url", "source_units_json", "batch_id", "ingested_at_utc",
    )
)

retail = (
    raw.where(F.col("dataset_name") == "eia_retail_sales")
    .withColumn("state_code", F.upper(F.trim("state_code_raw")))
    .withColumn("retail_sales_mwh", numeric_value("retail_sales_raw"))
    .withColumn("retail_sales_status", numeric_status("retail_sales_raw"))
    .join(F.broadcast(states), "state_code", "inner")
    .select(
        "period", "state_code", "state_name", "retail_sales_mwh", "retail_sales_status",
        "retail_sales_units_raw",
        "source_url", "source_units_json", "batch_id", "ingested_at_utc",
    )
)

population = (
    raw.where(F.col("dataset_name") == "census_population_vintage_2021")
    .withColumnRenamed("state_fips_raw", "census_state_fips")
    .withColumn("population", numeric_value("population_raw").cast("long"))
    .join(F.broadcast(states), "census_state_fips", "inner")
    .select(
        "state_code", "state_name", "census_state_fips", "population",
        F.lit(2021).alias("estimate_year"), F.lit("Vintage 2021").alias("vintage"),
        "source_url", "batch_id", "ingested_at_utc",
    )
)

In [ ]:
def assert_unique(df, keys, name):
    duplicates = df.groupBy(*keys).count().where(F.col("count") > 1)
    if duplicates.limit(1).count():
        duplicates.show(20, truncate=False)
        raise RuntimeError(f"{name} contains duplicate keys: {keys}")

def assert_nonnegative(df, columns, name):
    condition = None
    for column in columns:
        check = F.col(column).isNotNull() & (F.col(column) < 0)
        condition = check if condition is None else condition | check
    if df.where(condition).limit(1).count():
        raise RuntimeError(f"{name} contains negative reported values.")

assert_unique(operational, ["period", "state_code", "fuel_code"], "operational")
assert_unique(retail, ["period", "state_code"], "retail")
assert_unique(population, ["state_code"], "population")
assert_nonnegative(
    operational,
    ["generation_mwh", "total_consumption", "consumption_for_eg", "total_consumption_btu", "consumption_for_eg_btu"],
    "operational",
)
assert_nonnegative(retail, ["retail_sales_mwh"], "retail")
assert_nonnegative(population, ["population"], "population")

for name, df in {"operational": operational, "retail": retail}.items():
    coverage = df.groupBy("period").agg(F.countDistinct("state_code").alias("jurisdictions"))
    bad = coverage.where(F.col("jurisdictions") != expected_jurisdictions)
    if bad.limit(1).count():
        bad.show(truncate=False)
        raise RuntimeError(f"{name} does not cover all 50 states plus DC for every period.")
if population.select("state_code").distinct().count() != expected_jurisdictions:
    raise RuntimeError("Census population does not cover all 50 states plus DC.")

In [ ]:
selected_periods = sorted(row["period"] for row in retail.select("period").distinct().collect())
replace_where = f"period >= '{selected_periods[0]}' AND period <= '{selected_periods[-1]}'"
for table_name in ["silver.state_month_fuel", "silver.state_month_retail_sales"]:
    if spark.catalog.tableExists(table_name):
        keep = ",".join(f"'{period}'" for period in selected_periods)
        spark.sql(f"DELETE FROM {table_name} WHERE period NOT IN ({keep})")
(
    operational.write.format("delta").mode("overwrite")
    .option("replaceWhere", replace_where)
    .partitionBy("period").saveAsTable("silver.state_month_fuel")
)
(
    retail.write.format("delta").mode("overwrite")
    .option("replaceWhere", replace_where)
    .partitionBy("period").saveAsTable("silver.state_month_retail_sales")
)
population.write.format("delta").mode("overwrite").saveAsTable("silver.state_population")

spark.sql("OPTIMIZE silver.state_month_fuel ZORDER BY (state_code, fuel_code)")
spark.sql("OPTIMIZE silver.state_month_retail_sales ZORDER BY (state_code)")
print(
    {
        "batch_id": batch_id,
        "operational_rows": operational.count(),
        "retail_rows": retail.count(),
        "population_rows": population.count(),
    }
)